In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

OUT_DIR = '/home/weiji/restart_exam/plots/individual_test/'
os.makedirs(OUT_DIR, exist_ok=True)

def set_plot_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 9,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "savefig.dpi": 300,
        "figure.dpi": 300,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

# ===== 计算逻辑保持不变 =====
def calc_parc_o3(var, p_top=30, p_bottom=70):
    m_air = 28.964 / (6.022E23)
    g = 980.6
    if 'plev' in var.dims:
        plev = var.plev; level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev;  level = 'lev'
    elif 'level' in var.dims:
        plev = var.level; level = 'level'
    else:
        raise ValueError('No pressure level coordinate found in data.')
    time = var.time
    delta_p = np.zeros((len(time), len(plev)))
    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100; factor_2 = 1
    elif plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100; factor_2 = 1
    elif plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1;   factor_2 = 100
    elif plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1;   factor_2 = 100
    else:
        factor = 1;   factor_2 = 100
    if plev[0] < plev[-1]:
        for i in range(1, len(plev)):
            delta_p[:, i] = plev[i] - plev[i-1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_top * factor_2, p_bottom * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16
    else:
        for i in range(0, len(plev)-1):
            delta_p[:, i] = plev[i] - plev[i+1]
        weights_p = xr.DataArray(delta_p * factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)
        O3 = O3.sel(**{level: slice(p_bottom * factor_2, p_top * factor_2)})
        O3 = O3.sum(dim=level) / 2.687E16
    return O3.where(O3 != 0)

def spatial_average_coslat(O3_data, lat_start=60, lat_end=90):
    if 'lon' in O3_data.dims:
        O3_data = O3_data.mean(dim='lon')
    lat = O3_data['lat']
    if lat[0] <= lat[-1]:
        var = O3_data.sel(lat=slice(lat_start, lat_end))
    else:
        var = O3_data.sel(lat=slice(lat_end, lat_start))
    weights = np.cos(np.deg2rad(var['lat']))
    return var.weighted(weights).mean(dim='lat')

def load_member_timeseries(file_pattern):
    paths = sorted(glob.glob(file_pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {file_pattern}")
    print(f"[DEBUG] matched {len(paths)} files: e.g., {paths[0]}")
    ds = xr.open_mfdataset(paths, concat_dim='member', combine='nested')
    o3 = ds['O3']
    du = calc_parc_o3(o3, 30, 70)
    du_6090 = spatial_average_coslat(du).transpose('time', 'member')
    arr = du_6090.values
    print(f"[DEBUG] DU dims = {du_6090.dims}, shape = {tuple(du_6090.shape)}")
    print(f"[DEBUG] DU overall range: min={np.nanmin(arr):.2f}, max={np.nanmax(arr):.2f} DU")
    ds.close()
    return du_6090

def per_member_minima(du_ts, label="run"):
    results = []
    time = du_ts['time']
    doy = time.dt.dayofyear
    base_doy = 32
    for m in range(du_ts.sizes['member']):
        series = du_ts.isel(member=m).values
        if not np.isfinite(series).any():
            print(f"[WARN] {label}: member {m} all-NaN; skip.")
            continue
        idx = int(np.nanargmin(series))
        vmin = float(series[idx])
        if vmin < 70:
            smin = float(np.nanmin(series)); smax = float(np.nanmax(series)); smean = float(np.nanmean(series))
            print(f"[DEBUG-LOW] {label}: member={m} min={vmin:.2f} DU < 70; range [{smin:.2f},{smax:.2f}], mean={smean:.2f}")
        tmin = time.values[idx]
        d_since = int(doy.values[idx]) - base_doy
        results.append({
            'member': m,
            'min_DU': vmin,
            'date': str(tmin),
            'days_since_Feb1': d_since
        })
    print(f"[DEBUG] {label}: collected {len(results)} minima points")
    return results

def gather_xy(points_list):
    xs, ys = [], []
    for pts in points_list:
        for p in pts:
            if np.isfinite(p['days_since_Feb1']) and np.isfinite(p['min_DU']):
                xs.append(p['days_since_Feb1']); ys.append(p['min_DU'])
    if not xs:
        raise RuntimeError("No valid points to determine axis limits.")
    return (min(xs), max(xs)), (min(ys), max(ys))

def group_stats(points):
    vals = [p['min_DU'] for p in points if np.isfinite(p['min_DU'])]
    if len(vals) == 0:
        return np.nan, np.nan
    mean = float(np.nanmean(vals))
    std = float(np.nanstd(vals, ddof=1)) if len(vals) >= 2 else np.nan
    return mean, std






file_0008_Feb_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc'
file_0008_Mar_small = '/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc'
file_0008_Feb_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc'
file_0008_Mar_nocpl = '/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc'

print("[STEP] Load & reduce — Coupled Feb")
du_feb_c = load_member_timeseries(file_0008_Feb_small)
print("[STEP] Load & reduce — Coupled Mar")
du_mar_c = load_member_timeseries(file_0008_Mar_small)
print("[STEP] Load & reduce — NoCouple Feb")
du_feb_n = load_member_timeseries(file_0008_Feb_nocpl)
print("[STEP] Load & reduce — NoCouple Mar")
du_mar_n = load_member_timeseries(file_0008_Mar_nocpl)

print("[STEP] Compute minima — Coupled Feb")
feb_cpl_pts = per_member_minima(du_feb_c, label="Coupled-Feb")
print("[STEP] Compute minima — Coupled Mar")
mar_cpl_pts = per_member_minima(du_mar_c, label="Coupled-Mar")
print("[STEP] Compute minima — NoCouple Feb")
feb_nocpl_pts = per_member_minima(du_feb_n, label="NoCouple-Feb")
print("[STEP] Compute minima — NoCouple Mar")
mar_nocpl_pts = per_member_minima(du_mar_n, label="NoCouple-Mar")





In [ ]:
#绘制最小值/最小值日期
def plot_scatter(feb_cpl, feb_nocpl, mar_cpl, mar_nocpl, save_path):
    set_plot_style()
    try:
        plt.rcParams['hatch.linewidth'] = 1.0  # 提高 Mar 网纹可见性
    except Exception:
        pass

    fig, ax = plt.subplots(figsize=(4.8, 3.6))

    # ===== 动态范围 + 留白 =====
    (x_min, x_max), (y_min, y_max) = gather_xy([feb_cpl, feb_nocpl, mar_cpl, mar_nocpl])
    dx = max(1.0, x_max - x_min); dy = max(1.0, y_max - y_min)
    x_pad = max(1.0, 0.05 * dx);   y_pad = max(2.0, 0.05 * dy)
    x0, x1 = x_min - x_pad, x_max + x_pad
    y0, y1 = max(0.0, y_min - y_pad), y_max + y_pad
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)

    # ===== 颜色与样式 =====
    BLUE = '#1f77b4'
    RED  = '#d62728'
    FEB_FILL_ALPHA = 0.35
    SCAT_S = 15
    HATCH = '////'

    # ===== 数据统计 =====
    m_feb_c, s_feb_c = group_stats(feb_cpl)
    m_mar_c, s_mar_c = group_stats(mar_cpl)
    m_feb_n, s_feb_n = group_stats(feb_nocpl)
    m_mar_n, s_mar_n = group_stats(mar_nocpl)

    # ===== 阴影区与均值线 =====
    def draw_band_and_line(mean, std, color, linestyle, is_mar):
        if not np.isfinite(mean): return None
        if np.isfinite(std) and std > 0:
            if is_mar:
                rect = mpatches.Rectangle(
                    (x0, mean - std), (x1 - x0), (2 * std),
                    facecolor='none', edgecolor=color, hatch=HATCH,
                    linewidth=0.0, zorder=1.0
                )
                ax.add_patch(rect)
            else:
                ax.fill_between([x0, x1], mean - std, mean + std,
                                color=color, alpha=FEB_FILL_ALPHA, linewidth=0, zorder=1.0)
        ax.axhline(mean, color=color, linestyle=linestyle, linewidth=1.2, zorder=2.8)

    draw_band_and_line(m_feb_c, s_feb_c, BLUE, '-',  False)
    draw_band_and_line(m_mar_c, s_mar_c, BLUE, '--', True)
    draw_band_and_line(m_feb_n, s_feb_n, RED,  '-',  False)
    draw_band_and_line(m_mar_n, s_mar_n, RED,  '--', True)

    # ===== 散点 =====
    if feb_cpl:
        ax.scatter([p['days_since_Feb1'] for p in feb_cpl],
                   [p['min_DU'] for p in feb_cpl],
                   facecolor=BLUE, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='o', zorder=3.0)
    if feb_nocpl:
        ax.scatter([p['days_since_Feb1'] for p in feb_nocpl],
                   [p['min_DU'] for p in feb_nocpl],
                   facecolor=RED, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='o', zorder=3.0)
    if mar_cpl:
        ax.scatter([p['days_since_Feb1'] for p in mar_cpl],
                   [p['min_DU'] for p in mar_cpl],
                   facecolor=BLUE, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='^', zorder=3.0)
    if mar_nocpl:
        ax.scatter([p['days_since_Feb1'] for p in mar_nocpl],
                   [p['min_DU'] for p in mar_nocpl],
                   facecolor=RED, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='^', zorder=3.0)

    ax.set_xlabel('date of minimum polar ozone (in days since 1 February)')
    ax.set_ylabel('Minimum O₃ (DU, 30–70 hPa, 60–90°N)')
    ax.grid(True, linestyle='--', alpha=0.3)

    # ===== 图例：点 / 线 / 带 分开标记（带方框）=====
    def band_proxy(color, is_mar):
        if is_mar:
            return mpatches.Patch(facecolor='none', edgecolor=color, hatch=HATCH, linewidth=0.8)
        else:
            return mpatches.Patch(facecolor=color, edgecolor='none', alpha=FEB_FILL_ALPHA)

    def line_proxy(color, linestyle):
        return Line2D([0, 1], [0, 0], color=color, linestyle=linestyle, linewidth=1.1)

    def scatter_proxy(color, marker):
        return Line2D([0], [0], marker=marker, linestyle='None',
                      markerfacecolor=color, markeredgecolor='black', markersize=4.0)

    handles = [
        band_proxy(BLUE, False), line_proxy(BLUE, '-'),  scatter_proxy(BLUE, 'o'),
        band_proxy(RED,  False), line_proxy(RED,  '-'),  scatter_proxy(RED,  'o'),
        band_proxy(BLUE, True),  line_proxy(BLUE, '--'), scatter_proxy(BLUE, '^'),
        band_proxy(RED,  True),  line_proxy(RED,  '--'), scatter_proxy(RED,  '^'),
    ]
    labels = [
        'Feb couple — BAND (±1σ)', 'Feb couple — AVG', 'Feb couple — SCATTER',
        'Feb nocouple — BAND (±1σ)', 'Feb nocouple — AVG', 'Feb nocouple — SCATTER',
        'Mar couple — BAND (±1σ)', 'Mar couple — AVG', 'Mar couple — SCATTER',
        'Mar nocouple — BAND (±1σ)', 'Mar nocouple — AVG', 'Mar nocouple — SCATTER',
    ]

    legend = ax.legend(handles=handles, labels=labels,
                       loc='best',
                       fontsize=4.8, markerscale=0.65,    # 缩小 1/5
                       handlelength=1.0, handletextpad=0.35,
                       borderpad=0.15, labelspacing=0.15,
                       columnspacing=0.45, ncol=3,
                       frameon=True,       # 启用外框
                       framealpha=1.0,     # 不透明
                       edgecolor='black',  # 黑色边框
                       facecolor='white')  # 白底清晰

    plt.tight_layout()
    out_fig = os.path.join(OUT_DIR, 'FebMar_MinO3_Scatter_dynamic.png')
    plt.savefig(out_fig, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f"[INFO] Saved figure: {out_fig}")


plot_scatter(feb_cpl_pts, feb_nocpl_pts, mar_cpl_pts, mar_nocpl_pts,
                 save_path=None)  # save path在函数内统一

In [ ]:
# ===== Block3 (revised): Compute vortex breakup from U@10 hPa at 60°N (no 60–70 averaging) =====

def _select_level_hpa(da, target_hpa=10):
    """Robustly select ~target_hpa from Pa/hPa, both ascending/descending."""
    if   'plev'  in da.dims: level = 'plev'
    elif 'lev'   in da.dims: level = 'lev'
    elif 'level' in da.dims: level = 'level'
    else:
        raise ValueError("No pressure level coordinate among ['plev','lev','level'].")

    lev = da[level]
    tgt_val = target_hpa * 100.0 if float(lev.max()) > 1000 else target_hpa  # Pa vs hPa
    return da.sel(**{level: tgt_val}, method='nearest'), level

def load_u10_timeseries(file_pattern, lat_start=60, lat_end=70, target_hpa=10):
    """
    加载 U 场 → 取 10 hPa → 直接选最接近 60°N 的格点（不做 60–70 加权）→ 经向平均 → (time, member)
    注：保留原函数签名，但仅用 lat_start 作为目标纬度。
    """
    paths = sorted(glob.glob(file_pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {file_pattern}")
    print(f"[DEBUG] matched {len(paths)} files for U: e.g., {paths[0]}")

    ds = xr.open_mfdataset(paths, concat_dim='member', combine='nested')

    # 变量名兼容
    if 'U' in ds.data_vars:
        u = ds['U']
    elif 'ua' in ds.data_vars:
        u = ds['ua']
    else:
        ds.close()
        raise KeyError("Wind variable not found (tried 'U' and 'ua').")

    # 选 10 hPa
    u10, level_name = _select_level_hpa(u, target_hpa)

    # 纬度名兼容并选择最靠近 60N 的单点
    if   'lat' in u10.dims: lat_name = 'lat'
    elif 'latitude' in u10.dims: lat_name = 'latitude'
    else:
        ds.close()
        raise KeyError("Latitude coord not found (tried 'lat' and 'latitude').")

    # 直接选最近的 60°N
    u10_60N = u10.sel(**{lat_name: float(lat_start)}, method='nearest')

    # 经向平均
    if 'lon' in u10_60N.dims:
        u10_60N = u10_60N.mean(dim='lon')

    # 整形
    u10_60N = u10_60N.transpose('time', 'member')
    arr = u10_60N.values
    chosen_lat = float(u10_60N[lat_name].values) if lat_name in u10_60N.coords else float(lat_start)
    print(f"[DEBUG] U(10hPa,{chosen_lat:.2f}°N) dims={u10_60N.dims}, shape={tuple(u10_60N.shape)}")
    print(f"[DEBUG] U overall range: min={np.nanmin(arr):.2f}, max={np.nanmax(arr):.2f} m/s")

    ds.close()
    return u10_60N

def compute_breakup_dates(u10_ts, threshold=10.0, persist_days=5, label="run"):
    """
    定义：首次出现“连续 persist_days 天均 < threshold m/s”的起始日为极涡崩溃日期。
    输出：相对 2 月 1 日的天数（与前面保持一致）。
    """
    time = u10_ts['time']
    doy  = time.dt.dayofyear
    base_doy = 32  # Feb 1

    results = []
    for m in range(u10_ts.sizes['member']):
        series = u10_ts.isel(member=m).values.astype(float)

        if not np.isfinite(series).any():
            print(f"[WARN] {label}: member {m} U series all-NaN; skip.")
            continue

        cond = np.isfinite(series) & (series < threshold)  # 低于阈值即满足
        run = 0; start_idx = None
        for i, ok in enumerate(cond):
            if ok:
                run += 1
                if run >= persist_days:
                    start_idx = i - persist_days + 1
                    break
            else:
                run = 0

        if start_idx is None:
            print(f"[WARN] {label}: member {m} has no {persist_days}-day run < {threshold} m/s.")
            continue

        t0 = time.values[start_idx]
        d_since = int(doy.values[start_idx]) - base_doy
        results.append({'member': m, 'breakup_date': str(t0), 'days_since_Feb1': d_since})

    print(f"[DEBUG] {label}: collected {len(results)} breakup dates (persist_days={persist_days})")
    return results


# === 逐组计算（调用保持不变；内部已改为 60N 单点）===
print("[STEP] Load U@10hPa & compute breakup — Coupled Feb")
u10_feb_c = load_u10_timeseries(file_0008_Feb_small, lat_start=60, lat_end=70, target_hpa=10)
feb_cpl_bk = compute_breakup_dates(u10_feb_c, threshold=10.0, persist_days=5, label="Coupled-Feb")

print("[STEP] Load U@10hPa & compute breakup — Coupled Mar")
u10_mar_c = load_u10_timeseries(file_0008_Mar_small, lat_start=60, lat_end=70, target_hpa=10)
mar_cpl_bk = compute_breakup_dates(u10_mar_c, threshold=10.0, persist_days=5, label="Coupled-Mar")

print("[STEP] Load U@10hPa & compute breakup — NoCouple Feb")
u10_feb_n = load_u10_timeseries(file_0008_Feb_nocpl, lat_start=60, lat_end=70, target_hpa=10)
feb_nocpl_bk = compute_breakup_dates(u10_feb_n, threshold=10.0, persist_days=5, label="NoCouple-Feb")

print("[STEP] Load U@10hPa & compute breakup — NoCouple Mar")
u10_mar_n = load_u10_timeseries(file_0008_Mar_nocpl, lat_start=60, lat_end=70, target_hpa=10)
mar_nocpl_bk = compute_breakup_dates(u10_mar_n, threshold=10.0, persist_days=5, label="NoCouple-Mar")


In [ ]:
# ===== Block4: Plot scatter (X = breakup date since Feb 1, Y = min DU) =====

def _merge_minDU_and_breakup(min_pts, bk_pts):
    """按 member 合并：保留既有最小 O3 又有崩溃日期的成员。"""
    bk_map = {p['member']: p for p in bk_pts}
    merged = []
    miss = 0
    for p in min_pts:
        m = p['member']
        if m in bk_map:
            merged.append({
                'member': m,
                'min_DU': p['min_DU'],
                'bk_days_since_Feb1': bk_map[m]['days_since_Feb1']
            })
        else:
            miss += 1
    if miss:
        print(f"[INFO] merge: {miss} members skipped due to missing breakup date.")
    return merged

def _gather_xy_breakup(points_list):
    xs, ys = [], []
    for pts in points_list:
        for p in pts:
            x = p.get('bk_days_since_Feb1', np.nan)
            y = p.get('min_DU', np.nan)
            if np.isfinite(x) and np.isfinite(y):
                xs.append(x); ys.append(y)
    if not xs:
        raise RuntimeError("No valid points to determine axis limits for breakup scatter.")
    return (min(xs), max(xs)), (min(ys), max(ys))

def group_stats_from_minDU(points):
    vals = [p['min_DU'] for p in points if np.isfinite(p.get('min_DU', np.nan))]
    if len(vals) == 0:
        return np.nan, np.nan
    mean = float(np.nanmean(vals))
    std  = float(np.nanstd(vals, ddof=1)) if len(vals) >= 2 else np.nan
    return mean, std

def plot_scatter_breakup(feb_cpl, feb_nocpl, mar_cpl, mar_nocpl):
    set_plot_style()
    try:
        plt.rcParams['hatch.linewidth'] = 1.0
    except Exception:
        pass

    fig, ax = plt.subplots(figsize=(4.8, 3.6))

    # 动态范围
    (x_min, x_max), (y_min, y_max) = _gather_xy_breakup([feb_cpl, feb_nocpl, mar_cpl, mar_nocpl])
    dx = max(1.0, x_max - x_min); dy = max(1.0, y_max - y_min)
    x_pad = max(1.0, 0.05 * dx);   y_pad = max(2.0, 0.05 * dy)
    x0, x1 = x_min - x_pad, x_max + x_pad
    y0, y1 = max(0.0, y_min - y_pad), y_max + y_pad
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)

    # 样式
    BLUE = '#1f77b4'
    RED  = '#d62728'
    FEB_FILL_ALPHA = 0.35
    SCAT_S = 15
    HATCH = '////'

    # 仅对 Y（min DU）做均值±1σ带
    m_feb_c, s_feb_c = group_stats_from_minDU(feb_cpl)
    m_mar_c, s_mar_c = group_stats_from_minDU(mar_cpl)
    m_feb_n, s_feb_n = group_stats_from_minDU(feb_nocpl)
    m_mar_n, s_mar_n = group_stats_from_minDU(mar_nocpl)

    def draw_band_and_line(mean, std, color, linestyle, is_mar):
        if not np.isfinite(mean): return
        if np.isfinite(std) and std > 0:
            if is_mar:
                rect = mpatches.Rectangle(
                    (x0, mean - std), (x1 - x0), (2 * std),
                    facecolor='none', edgecolor=color, hatch=HATCH,
                    linewidth=0.0, zorder=1.0
                )
                ax.add_patch(rect)
            else:
                ax.fill_between([x0, x1], mean - std, mean + std,
                                color=color, alpha=FEB_FILL_ALPHA, linewidth=0, zorder=1.0)
        ax.axhline(mean, color=color, linestyle=linestyle, linewidth=1.2, zorder=2.8)

    draw_band_and_line(m_feb_c, s_feb_c, BLUE, '-',  False)
    draw_band_and_line(m_mar_c, s_mar_c, BLUE, '--', True)
    draw_band_and_line(m_feb_n, s_feb_n, RED,  '-',  False)
    draw_band_and_line(m_mar_n, s_mar_n, RED,  '--', True)

    # 散点（X=breakup 天数, Y=min DU）
    if feb_cpl:
        ax.scatter([p['bk_days_since_Feb1'] for p in feb_cpl],
                   [p['min_DU'] for p in feb_cpl],
                   facecolor=BLUE, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='o', zorder=3.0)
    if feb_nocpl:
        ax.scatter([p['bk_days_since_Feb1'] for p in feb_nocpl],
                   [p['min_DU'] for p in feb_nocpl],
                   facecolor=RED, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='o', zorder=3.0)
    if mar_cpl:
        ax.scatter([p['bk_days_since_Feb1'] for p in mar_cpl],
                   [p['min_DU'] for p in mar_cpl],
                   facecolor=BLUE, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='^', zorder=3.0)
    if mar_nocpl:
        ax.scatter([p['bk_days_since_Feb1'] for p in mar_nocpl],
                   [p['min_DU'] for p in mar_nocpl],
                   facecolor=RED, edgecolor='black', s=SCAT_S, alpha=0.9,
                   marker='^', zorder=3.0)

    ax.set_xlabel('vortex breakup date (days since 1 February)')
    ax.set_ylabel('Minimum O₃ (DU, 30–70 hPa, 60–90°N)')
    ax.grid(True, linestyle='--', alpha=0.3)

    # 图例：与 Block2 保持风格与顺序
    def band_proxy(color, is_mar):
        if is_mar:
            return mpatches.Patch(facecolor='none', edgecolor=color, hatch=HATCH, linewidth=0.8)
        else:
            return mpatches.Patch(facecolor=color, edgecolor='none', alpha=FEB_FILL_ALPHA)

    def line_proxy(color, linestyle):
        return Line2D([0, 1], [0, 0], color=color, linestyle=linestyle, linewidth=1.1)

    def scatter_proxy(color, marker):
        return Line2D([0], [0], marker=marker, linestyle='None',
                      markerfacecolor=color, markeredgecolor='black', markersize=4.0)

    handles = [
        band_proxy(BLUE, False), line_proxy(BLUE, '-'),  scatter_proxy(BLUE, 'o'),
        band_proxy(RED,  False), line_proxy(RED,  '-'),  scatter_proxy(RED,  'o'),
        band_proxy(BLUE, True),  line_proxy(BLUE, '--'), scatter_proxy(BLUE, '^'),
        band_proxy(RED,  True),  line_proxy(RED,  '--'), scatter_proxy(RED,  '^'),
    ]
    labels = [
        'Feb couple — BAND (±1σ)', 'Feb couple — AVG', 'Feb couple — SCATTER',
        'Feb nocouple — BAND (±1σ)', 'Feb nocouple — AVG', 'Feb nocouple — SCATTER',
        'Mar couple — BAND (±1σ)', 'Mar couple — AVG', 'Mar couple — SCATTER',
        'Mar nocouple — BAND (±1σ)', 'Mar nocouple — AVG', 'Mar nocouple — SCATTER',
    ]

    legend = ax.legend(handles=handles, labels=labels,
                       loc='best',
                       fontsize=4.8, markerscale=0.65,
                       handlelength=1.0, handletextpad=0.35,
                       borderpad=0.15, labelspacing=0.15,
                       columnspacing=0.45, ncol=3,
                       frameon=True, framealpha=1.0, edgecolor='black', facecolor='white')

    plt.tight_layout()
    out_fig = os.path.join(OUT_DIR, 'FebMar_MinO3_vs_Breakup_Scatter.png')
    plt.savefig(out_fig, bbox_inches='tight', dpi=300)
    plt.close(fig)
    print(f"[INFO] Saved figure: {out_fig}")


# === 合并最小 DU 与崩溃日期（按成员对齐） ===
feb_cpl_merged   = _merge_minDU_and_breakup(feb_cpl_pts,   feb_cpl_bk)
feb_nocpl_merged = _merge_minDU_and_breakup(feb_nocpl_pts, feb_nocpl_bk)
mar_cpl_merged   = _merge_minDU_and_breakup(mar_cpl_pts,   mar_cpl_bk)
mar_nocpl_merged = _merge_minDU_and_breakup(mar_nocpl_pts, mar_nocpl_bk)

# === 绘图 ===
plot_scatter_breakup(feb_cpl_merged, feb_nocpl_merged, mar_cpl_merged, mar_nocpl_merged)
